# Regex project – MagicBricks property scraping
Same flow as the CarDekho notebook: Selenium + BeautifulSoup + regex → DataFrame → CSV → cleaning → EDA.

In [ ]:
!pip install selenium

In [ ]:
import pandas as pd

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import re
import time

TARGET = 1000          # records you want
CITIES = ['Hyderabad', 'Bangalore', 'Mumbai', 'Pune', 'Chennai', 'New-Delhi', 'Gurgaon', 'Noida']
BHKS = ['1', '2', '3', '4']   # one search per city per BHK = more listings per city

BASE = 'https://www.magicbricks.com/property-for-sale/residential-real-estate?bedroom={b}&cityName={c}'
urls = [(c, b, BASE.format(b=b, c=c)) for c in CITIES for b in BHKS]

opts = webdriver.ChromeOptions()
opts.add_argument('--start-maximized')
opts.add_argument('--disable-blink-features=AutomationControlled')
driver = webdriver.Chrome(options=opts)

title_re = re.compile(r'(\d+)\s*BHK\s*(.{0,40}?)\s*for\s*sale\s*in\s*([^,|\n]+)', re.I)
price_re = re.compile(r'₹\s*([\d.,]+)\s*(Lac|Lakh|Cr)\b', re.I)
pps_re   = re.compile(r'₹\s*([\d,]+)\s*(?:/|per)\s*sq\.?\s*ft', re.I)
area_re  = re.compile(r'(Carpet|Super|Built[- ]?up)\s*Area\s*:?\s*([\d,.]+)\s*sq\.?\s*ft', re.I)
furn_re  = re.compile(r'(Unfurnished|Semi-Furnished|Furnished)', re.I)
stat_re  = re.compile(r'(Ready to Move|Under Construction)', re.I)

rows = []


def to_float(s):
    try:
        return float(s.replace(',', ''))
    except Exception:
        return None


def parse_card(card, city_name):
    # try the dedicated title element first, fall back to the whole card text
    try:
        title = card.find_element(By.CSS_SELECTOR, '.mb-srp__card--title').text
    except Exception:
        title = ''
    text = card.text.replace('\n', ' ')
    m = title_re.search(title) or title_re.search(text)
    p = price_re.search(text)
    if not m or not p:
        return None

    pps = pps_re.search(text)
    ar = area_re.search(text)
    fu = furn_re.search(text)
    st = stat_re.search(text)

    return {
        'City': city_name,
        'BHK': int(m.group(1)),
        'Property_Type': m.group(2).strip() or None,
        'Locality': m.group(3).strip(),
        'Price': to_float(p.group(1)),
        'Price_Unit': p.group(2),
        'Price_per_sqft': to_float(pps.group(1)) if pps else None,
        'Area_sqft': to_float(ar.group(2)) if ar else None,
        'Area_Type': ar.group(1) if ar else None,
        'Furnishing': fu.group(1) if fu else None,
        'Status': st.group(1) if st else None,
    }


def collect(city_name, done):
    # duplicates are NOT removed: every listing card is kept.
    # 'done' only remembers which cards on THIS page were already read,
    # so the same card is not added again every time we scroll.
    new = 0
    for card in driver.find_elements(By.CSS_SELECTOR, 'div.mb-srp__card'):
        if card.id in done:
            continue
        done.add(card.id)
        try:
            rec = parse_card(card, city_name)
        except Exception:
            continue
        if not rec:
            continue
        rows.append(rec)
        new += 1
    return new


per_search = -(-TARGET // len(urls)) + 30    # share of the target for each search

for city_name, b, url in urls:
    if len(rows) >= TARGET:
        break
    print(f'\n{city_name} | {b} BHK | rows so far: {len(rows)}')
    driver.get(url)
    time.sleep(5)

    done = set()
    got = 0
    idle = 0
    for step in range(200):
        # scroll in small steps so lazy loading triggers
        for _ in range(4):
            driver.execute_script('window.scrollBy(0, 900);')
            time.sleep(0.6)

        # click "load more" style buttons if present
        for el in driver.find_elements(By.XPATH, "//*[self::a or self::button or self::div][contains(translate(normalize-space(text()),'LOADMORE','loadmore'),'load more')]"):
            try:
                driver.execute_script('arguments[0].click();', el)
                time.sleep(2)
            except Exception:
                pass

        added = collect(city_name, done)    # parse every round, so nothing is lost if cards get recycled
        got += added
        idle = 0 if added else idle + 1
        print(f'  scroll {step + 1}: +{added} (this search: {got})')

        if got >= per_search or idle >= 8:
            break

    pd.DataFrame(rows).to_csv('property_data_partial.csv', index=False)

driver.quit()

df = pd.DataFrame(rows)
print('Total records:', len(df))
df.to_csv('property_data.csv', index=False)
print(df.head())

In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
print("rows and columns:", df.shape)
df.head()

#### Data Cleaning

In [ ]:
# column names -> lowercase
df.columns = df.columns.str.lower()

# text values -> lowercase + trim spaces
text_columns = ["city", "property_type", "locality", "price_unit", "area_type", "furnishing", "status"]
for col in text_columns:
    df[col] = df[col].astype("string").str.strip().str.lower()

# numbers were scraped as text (e.g. "1,250") -> convert to numeric
for col in ["bhk", "area_sqft", "price"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")

# price is in lakh or crore -> convert everything to rupees
df["price_inr"] = np.where(df["price_unit"] == "cr", df["price"] * 10000000, df["price"] * 100000)

# drop the columns we no longer need (price_per_sqft is rebuilt below)
df = df.drop(columns=["price", "price_unit", "price_per_sqft"])

print("columns now:", list(df.columns))
df.head()

In [ ]:
print("rows before cleaning:", len(df))

#  remove rows with no area or no price
df = df.dropna(subset=["area_sqft", "price_inr"])

#  keep only sensible home sizes and prices
df = df[(df["area_sqft"] >= 200) & (df["area_sqft"] <= 10000)]
df = df[(df["price_inr"] >= 500000) & (df["price_inr"] <= 500000000)]   # 5 lakh to 50 crore
df = df.copy()

#  new column: price per square foot
df["price_per_sqft"] = df["price_inr"] / df["area_sqft"]
df = df[(df["price_per_sqft"] >= 1000) & (df["price_per_sqft"] <= 100000)]

#  price in crores (easier to read in charts)
df["price_cr"] = df["price_inr"] / 10000000

df = df.reset_index(drop=True)

print("rows after cleaning :", len(df))
df.head()

In [ ]:
df.info()
print("\nmissing values per column:")
print(df.isna().sum())

#### Univariate Analysis

In [ ]:
df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].describe().round(2)

In [ ]:
# the three "averages" for price, explained simply
print("PRICE (in crores):")
print("  mean   (simple average) :", round(df["price_cr"].mean(), 2))
print("  median (middle value)   :", round(df["price_cr"].median(), 2))
print("  std    (spread)         :", round(df["price_cr"].std(), 2))

print("\nAREA (sqft):")
print("  mean   :", round(df["area_sqft"].mean()))
print("  median :", round(df["area_sqft"].median()))

print("\nMost common BHK (mode):", int(df["bhk"].mode()[0]))

In [ ]:
# counts for each text column (one at a time)
for col in ["city", "property_type", "furnishing", "status"]:
    print("----", col, "----")
    print(df[col].value_counts())
    print()

In [ ]:
# the same as percentages (share of each city)
(df["city"].value_counts(normalize=True) * 100).round(1)

### Bivariate Analysis

In [ ]:
# typical (median) price and price-per-sqft for each city
city_summary = (df.groupby("city")[["price_cr", "price_per_sqft"]]
                  .median().round(2)
                  .sort_values("price_cr", ascending=False))
city_summary

In [ ]:
# how price and size grow with the number of bedrooms (1-6 BHK homes)
homes = df[df["bhk"].between(1, 6)]
homes.groupby("bhk")[["price_cr", "area_sqft"]].mean().round(2)

In [ ]:
# does furnishing relate to price? (median price by furnishing type)
df.groupby("furnishing")["price_cr"].median().round(2).sort_values(ascending=False)

In [ ]:
# correlation: +1 = move together strongly, 0 = no link, -1 = opposite
df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].corr().round(2)

In [ ]:
# how many homes are 'ready to move' vs 'under construction' in each city
pd.crosstab(df["city"], df["status"])

#### Univariate Analysis(Charts)

In [ ]:
# how price is spread (in crores)
plt.figure(figsize=(8, 4))
sns.histplot(df["price_cr"], bins=50)
plt.title("Distribution of property price")
plt.xlabel("price (crores)")
plt.ylabel("number of homes")
plt.show()

In [ ]:
# how size is spread
plt.figure(figsize=(8, 4))
sns.histplot(df["area_sqft"], bins=50, color="seagreen")
plt.title("Distribution of area")
plt.xlabel("area (sqft)")
plt.ylabel("number of homes")
plt.show()

In [ ]:
# boxplot of price - the box is the middle 50%, dots are expensive outliers
plt.figure(figsize=(8, 3))
sns.boxplot(x=df["price_cr"], color="salmon")
plt.title("Price spread and outliers")
plt.xlabel("price (crores)")
plt.show()

In [ ]:
# count of homes per city and per BHK
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="city", ax=axes[0],
              order=df["city"].value_counts().index)
axes[0].set_title("Homes per city")
axes[0].tick_params(axis="x", rotation=45)

homes = df[df["bhk"].between(1, 6)]
sns.countplot(data=homes, x="bhk", ax=axes[1])
axes[1].set_title("Homes per BHK")

plt.tight_layout()
plt.show()

In [ ]:
# count of homes per furnishing and per status
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="furnishing", ax=axes[0],
              order=df["furnishing"].value_counts().index)
axes[0].set_title("Furnishing")
axes[0].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="status", ax=axes[1],
              order=df["status"].value_counts().index)
axes[1].set_title("Status")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# count of homes per property type
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="property_type", order=df["property_type"].value_counts().index)
plt.title("Homes per property type")
plt.xlabel("")
plt.xticks(rotation=20)
plt.show()

#### Bivariate Analysis(Charts)

In [ ]:
# do bigger homes cost more? (area vs price)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="area_sqft", y="price_cr", alpha=0.3)
plt.title("Area vs Price (each dot is one home)")
plt.xlabel("area (sqft)")
plt.ylabel("price (crores)")
plt.show()

In [ ]:
# price by city - which cities are pricier?
order = df.groupby("city")["price_cr"].median().sort_values(ascending=False).index
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="city", y="price_cr", order=order)
plt.title("Price by city")
plt.xlabel("")
plt.ylabel("price (crores)")
plt.ylim(0, 8)               # zoom in (hide extreme outliers for a clearer view)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# price by number of bedrooms
homes = df[df["bhk"].between(1, 6)]
plt.figure(figsize=(8, 5))
sns.boxplot(data=homes, x="bhk", y="price_cr")
plt.title("Price by BHK")
plt.xlabel("BHK (bedrooms)")
plt.ylabel("price (crores)")
plt.ylim(0, 8)
plt.tight_layout()
plt.show()

In [ ]:
# average price per sqft by city (the fairest way to compare cities)
order = df.groupby("city")["price_per_sqft"].median().sort_values(ascending=False).index
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="city", y="price_per_sqft", order=order, estimator=np.median, errorbar=None)
plt.title("Median price per sqft by city")
plt.xlabel("")
plt.ylabel("price per sqft (₹)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# all correlations in one picture (darker/brighter = stronger link)
plt.figure(figsize=(6, 5))
corr = df[["bhk", "area_sqft", "price_inr", "price_per_sqft"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation heatmap")
plt.show()